In [109]:
import random
from collections import deque

In [110]:
class Tile:
    
    def __init__(self, tuple_tile: tuple[int, int]):
        self.left_side = tuple_tile[0]
        self.right_side = tuple_tile[1]


    def __str__(self) -> str:
        return f"[{self.left_side}|{self.right_side}]"


    def __repr__(self) -> str:
        return self.__str__()


    def __eq__(self, other: Tile) -> bool:
        if not isinstance(other, Tile):
            return NotImplemented
        return (
            (self.left_side == other.left_side 
             and self.right_side == other.right_side)
            or 
            (self.left_side == other.right_side 
             and self.right_side == other.left_side)
        )


    def __iter__(self):
        yield self.left_side
        yield self.right_side


    def __add__(self, other):
        if isinstance(other, Tile):
            return (
                self.left_side + self.right_side
                + other.left_side + other.right_side
            )
        return NotImplemented


    @staticmethod
    def swap_tile(tile: Tile) -> Tile:
        tile.left_side, tile.right_side = tile.right_side, tile.left_side
        return tile

In [111]:
class Player:

    def __init__(self, name: str, tiles: list[Tile]):
        self.name = name
        self.tiles = tiles


    def __str__(self):
        return f"{self.name}\n{self.tiles}"


    def __repr__(self) -> str:
        return f"{self.name}\n{self.tiles}"

In [112]:
class Table:

    def __init__(self):
        self.tiles_deque = deque()


    def __str__(self):
        return list(self.tiles_deque)


    def add_first_tile(self, initial_tile: Tile):
        self.table_left_side = initial_tile.left_side
        self.table_right_side = initial_tile.right_side

        self.tiles_deque.append(initial_tile)


    def add_left(self, tile: Tile) -> None:
        if tile.right_side != self.table_left_side:
            if tile.left_side != self.table_left_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.appendleft(tile)
        self.table_left_side = tile.left_side


    def add_right(self, tile: Tile) -> None:
        if tile.left_side != self.table_right_side:
            if tile.right_side != self.table_right_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.append(tile)
        self.table_right_side = tile.right_side


    def show_table(self):
        print(list(self.tiles_deque))


In [ ]:
class Domino:

    def __init__(self):
            self.players = []
            self.sleep = None

            self.table = Table()

            self.tiles = Domino.make_tiles()
            self.TOTAL_TILES = len(self.tiles)


    @staticmethod
    def make_tiles():
        tiles_tup = []
        pivot = 0
        for side1 in range(0, 7):
            for side2 in range(pivot, 7):
                tiles_tup.append((side1, side2))
            pivot+=1
        tiles = [Tile(tup) for tup in tiles_tup]
        return tiles


    def divide_tiles(
            self,
            n_players = 4, 
            tile_count = 7,
            has_sleep = True
        ) -> list[Player]:
        
        if 4 < n_players or n_players < 2:
            raise ValueError("Só é permitido entre 2 a 4 jogadores")
        if tile_count < 5 or len(self.tiles) // n_players < tile_count:
            raise ValueError("Selecione uma quantidade válida de peças")
        
        random.shuffle(self.tiles)

        self.players = [
            Player(
                f"P{n+1}", 
                [self.tiles.pop() for _ in range(tile_count)]
            ) for n in range(n_players)]

        if not has_sleep or len(self.tiles) == 1:
            self.sleep = None
        else:
            self.sleep = self.tiles.copy()
            self.tiles.clear()

        return self.players


    def verify_starter(self) -> tuple[Player, Tile]:

        doubles = [Tile((n, n)) for n in range(7)][::-1]

        for double in doubles:
            for player in self.players:
                if double in player.tiles:
                    return player, double

        # se todas as carroças tiverem no dorme ele vai pegar o player com a maior peça
        tiles_filtered = sorted([t for t in Domino.make_tiles() if t.left_side != t.right_side], key=lambda x: sum(x), reverse=True)
        
        for tile in tiles_filtered:
            for player in self.players:
                if tile in player.tiles:
                    return player, tile

        # só pq o pylance tava reclamando
        player_rand = random.choice(self.players)
        return player_rand, max(player_rand.tiles, key=lambda tile: tile.left_side + tile.right_side)


In [114]:
domino = Domino()

players = domino.divide_tiles(
    n_players = 2, 
    tile_count = 5,
    has_sleep = True
    )

sleep = domino.sleep

for p in players:
    print(p)

print(f"Sleep: {sleep}\nSleep Count: {len(sleep) if sleep is not None else 0}")



P1
[[1|5], [5|6], [1|4], [0|6], [2|5]]
P2
[[4|6], [4|4], [0|4], [3|5], [2|4]]
Sleep: [[2|2], [5|5], [0|3], [0|5], [4|5], [2|3], [0|2], [0|1], [2|6], [1|6], [6|6], [0|0], [1|1], [3|6], [1|3], [1|2], [3|3], [3|4]]
Sleep Count: 18


In [116]:
starter, selected_tile = domino.verify_starter()
print(f"{starter}\nSelected tile:{selected_tile}")

P2
[[4|6], [4|4], [0|4], [3|5], [2|4]]
Selected tile:[4|4]


In [117]:
#TODO: fazer o player jogar; verificar vencedor e decidir vencedor caso o jogo tranque; deixar menos feio; TREINAR A IA
